# 64 — Documentary Site-Year Linkage

Links documentary evidence to configured repository sites using explicit
names, aliases, and conservative matching.

In [ ]:
import os, json, hashlib, platform, sys, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(path), "status": "empty_file"}
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def safe_read_parquet(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        df = pd.read_parquet(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest: dict, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def standardise_pollutant(x):
    s = str(x).strip().lower()
    mapping = {
        "oxides of nitrogen": "NOx",
        "nox": "NOx",
        "no2": "NO2",
        "no": "NO",
        "particulates": "Particulates",
        "dust": "Particulates",
        "pm10": "PM10",
        "pm2.5": "PM2.5",
        "sulphur dioxide": "SO2",
        "so2": "SO2",
        "hydrogen chloride": "HCl",
        "hcl": "HCl",
        "total organic carbon": "TOC",
        "toc": "TOC",
        "carbon monoxide": "CO",
        "co": "CO",
        "ammonia": "NH3",
        "nh3": "NH3",
    }
    return mapping.get(s, str(x).strip())

In [ ]:
step = "64_documentary_site_year_linkage"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

run_cfg = load_yaml(CONFIGS / "run.yml")
doc_cfg = load_yaml(CONFIGS / "documentary_sources.yml")
sites = pd.DataFrame(run_cfg.get("sites", []))
if sites.empty:
    raise RuntimeError("No sites found in configs/run.yml")

emissions, _ = safe_read_csv(OUTPUTS / "62_emissions_documentary_layer" / "site_year_pollutant_summary.csv")
cems, _ = safe_read_csv(OUTPUTS / "63_cems_noncompliance_layer" / "candidate_noncompliance_events.csv")
catalog, _ = safe_read_csv(DATA / "incinerator_report_catalog.csv")

aliases = doc_cfg.get("site_aliases", {})

sites["site_id"] = sites.get("id", pd.Series(dtype=str)).astype(str)
sites["site_name"] = sites.get("name", pd.Series(dtype=str)).astype(str)

alias_rows = []
for _, row in sites.iterrows():
    base_keys = {slugify(row["site_name"]), slugify(row["site_id"])}
    extra = set(slugify(x) for x in aliases.get(str(row["site_id"]), []))
    for k in base_keys | extra:
        alias_rows.append({"site_id": row["site_id"], "repo_site_name": row["site_name"], "match_key": k})
alias_df = pd.DataFrame(alias_rows).drop_duplicates()

def attach(df, name_col="site_name"):
    if df.empty:
        return pd.DataFrame()
    work = df.copy()
    work["match_key"] = work[name_col].map(slugify)
    return work.merge(alias_df, on="match_key", how="left")

linked_emissions = attach(emissions, "site_name") if not emissions.empty else pd.DataFrame()
linked_cems = attach(cems, "site_name") if not cems.empty else pd.DataFrame()
linked_catalog = attach(catalog, "site_name") if ("site_name" in catalog.columns and not catalog.empty) else pd.DataFrame()

linkage_summary = []
for sid, grp in alias_df.groupby("site_id"):
    linkage_summary.append({
        "site_id": sid,
        "alias_count": int(len(grp)),
        "emissions_linked_rows": int((linked_emissions["site_id"] == sid).sum()) if not linked_emissions.empty and "site_id" in linked_emissions.columns else 0,
        "cems_linked_rows": int((linked_cems["site_id"] == sid).sum()) if not linked_cems.empty and "site_id" in linked_cems.columns else 0,
        "catalog_linked_rows": int((linked_catalog["site_id"] == sid).sum()) if not linked_catalog.empty and "site_id" in linked_catalog.columns else 0,
    })
linkage_summary = pd.DataFrame(linkage_summary)

if not linked_emissions.empty: linked_emissions.to_csv(out / "linked_emissions.csv", index=False)
if not linked_cems.empty: linked_cems.to_csv(out / "linked_cems.csv", index=False)
if not linked_catalog.empty: linked_catalog.to_csv(out / "linked_catalog.csv", index=False)
linkage_summary.to_csv(out / "linkage_summary.csv", index=False)

manifest = manifest_base(step, [CONFIGS / "run.yml", CONFIGS / "documentary_sources.yml"])
for fn in ["linked_emissions.csv","linked_cems.csv","linked_catalog.csv","linkage_summary.csv"]:
    add_artifact(manifest, out / fn)
write_json(out / "manifest.json", manifest)

display(linkage_summary)